# Advanced AI Lab 1 — Sentiment Analysis

This is the **single entry point** for all Lab 1 experiments.  
Run the cells top-to-bottom in each section you need.  
All heavy logic lives in the project files — this notebook just calls them.

| Section | Task |
|---|---|
| 0 — Setup | Check GPU, install dependencies, wandb login |
| 1 — Task 1.1 ANN | Simple ANN on small (1 K) and large (25 K) datasets |
| 2 — Task 1.1 BiLSTM | Bidirectional LSTM on small and large datasets |
| 3 — Task 1.2 BERT | Fine-tune BERT on the large Amazon dataset |
| 4 — Task 1.2 DistilBERT | Fine-tune DistilBERT on the large Amazon dataset |
| 5 — Grade 5 | BERT + DistilBERT on the public ~1 GB amazon_polarity dataset |
| 6 — Comparison | Side-by-side results across all models |
| 7 — W&B | View training curves at https://wandb.ai |

---
## Section 0 — Setup

Run this cell once before anything else.  
It verifies PyTorch, shows the GPU (if available), and confirms all modules load correctly.

In [1]:
import sys
import os

# Make sure the Lab 1 root is on the Python path so all imports resolve
project_root = os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, project_root)

import torch
import config

print("=" * 55)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print(f"Device in use   : {config.DEVICE}")
print("=" * 55)
print()
print("Datasets found:")
print(f"  Small (1 K)  : {os.path.exists(config.SMALL_DATASET_PATH)}  →  {config.SMALL_DATASET_PATH}")
print(f"  Large (25 K) : {os.path.exists(config.LARGE_DATASET_PATH)}  →  {config.LARGE_DATASET_PATH}")
print()
print("All project modules are ready.")
print("Run any section below to start an experiment.")

PyTorch version : 2.11.0+cu130
CUDA available  : True
GPU             : NVIDIA GeForce RTX 2080 Ti
Device in use   : cuda

Datasets found:
  Small (1 K)  : True  →  /Labs/Lab1/amazon_cells_labelled.txt
  Large (25 K) : True  →  /Labs/Lab1/amazon_cells_labelled_LARGE_25K.txt

All project modules are ready.
Run any section below to start an experiment.


### Section 0.1 — Weights & Biases Login (run once)

All experiments log to **https://wandb.ai** automatically.  
Run the cell below to install wandb (if missing) and authenticate.  
Get your API key at **https://wandb.ai/authorize**

In [2]:
import subprocess, sys

import wandb
wandb.login()   # prompts for API key if not already stored

print()
print("Logged in to wandb.")
print("After running experiments, view all results at: https://wandb.ai")
print(f"Project name: {__import__('config').WANDB_PROJECT}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: deeplab15group (deeplab15group-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Logged in to wandb.
After running experiments, view all results at: https://wandb.ai
Project name: advanced-ai-lab-1


---
## Section 1 — Task 1.1: Simple ANN

A feed-forward network trained on **TF-IDF bag-of-words features**.  
We run the same architecture twice — once on the 1 K small dataset, once on the 25 K large dataset — to observe how training-data volume impacts accuracy.

| Cell | Dataset | What we learn |
|---|---|---|
| 1.1 | Small (1 K) | Baseline — limited data |
| 1.2 | Large (25 K) | Effect of 25× more training data |

In [3]:
# 1.1  Simple ANN — small dataset (1 K reviews)
from experiments.task01_ann import main as run_ann

ann_small = run_ann(dataset="small")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading ANN data  [small dataset] …
  Preprocessing text (classical) …
  Fitting TF-IDF on training split (will NOT see val / test) …
  TF-IDF vocab: 4,033 features
  Split: 700 train / 150 val / 150 test

Building ANN  (vocab_size=4,033) …
  [build_ann] dataset='small' → SmallANN (2 blocks, 128→64)
  Total parameters    : 524,994
  Trainable parameters: 524,994



══════════════════════════════════════════════════════════════
  Experiment  : Task01_ANN_Small
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=0.0005
  Epochs      : 50
  Patience    : 3
  Trainable   : 524,994 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/50  |  train  loss=0.7163  acc=50.9%  |  val  loss=0.6931  acc=50.0%  f1=0.6606  prec=0.5000  rec=0.9733


  Epoch  2/50  |  train  loss=0.6865  acc=56.1%  |  val  loss=0.6741  acc=56.7%  f1=0.6632  prec=0.5424  rec=0.8533


  Epoch  3/50  |  train  loss=0.6364  acc=64.6%  |  val  loss=0.6316  acc=72.0%  f1=0.7407  prec=0.6897  rec=0.8000


...

  Epoch  8/50  |  train  loss=0.0740  acc=98.6%  |  val  loss=0.4922  acc=78.0%  f1=0.7925  prec=0.7500  rec=0.8400


  Epoch  9/50  |  train  loss=0.0662  acc=98.9%  |  val  loss=0.4925  acc=77.3%  f1=0.7901  prec=0.7356  rec=0.8533
  Early stopping triggered (no improvement for 3 epochs).



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 73.33%   F1 : 0.7436   Precision : 0.7160   Recall : 0.7733
──────────────────────────────────────────────────────────────



batch_loss,▇██▇███▇▇▇▇▆▆▆▇▇▆▅▅▆▅▅▅▄▃▃▃▂▂▄▂▂▂▁▂▁▁▁▁▁
epoch,▁▂▃▄▅▅▆▇█
train_accuracy,▁▂▃▅▆▇███
train_loss,██▇▆▅▃▂▁▁
val_accuracy,▁▃▇▇█████
val_f1,▁▁▅▅▇█▇██
val_loss,█▇▆▄▂▁▁▂▂
val_precision,▁▂▆▇█████
val_recall,█▄▂▁▂▃▂▃▄
batch_loss,0.04079
epoch,9


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_ANN_Small.pth
[Result] Experiment       : Task01_ANN_Small
[Result] Best Val Accuracy: 78.00%
[Result] Test Accuracy    : 73.33%
[Result] Test F1          : 0.7436



In [4]:
# 1.2  Simple ANN — large dataset (25 K reviews)
from experiments.task01_ann import main as run_ann

ann_large = run_ann(dataset="large")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading ANN data  [large dataset] …
  Preprocessing text (classical) …
  Fitting TF-IDF on training split (will NOT see val / test) …
  TF-IDF vocab: 11,619 features
  Split: 17,510 train / 3,740 val / 3,750 test

Building ANN  (vocab_size=11,619) …
  [build_ann] dataset='large' → LargeANN (3 blocks, 512→256→64)
  Total parameters    : 6,098,882
  Trainable parameters: 6,098,882



══════════════════════════════════════════════════════════════
  Experiment  : Task01_ANN_Large
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=5e-05
  Epochs      : 20
  Patience    : 2
  Trainable   : 6,098,882 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/20  |  train  loss=0.6539  acc=60.3%  |  val  loss=0.5606  acc=75.5%  f1=0.8268  prec=0.7222  rec=0.9668


  Epoch  2/20  |  train  loss=0.4810  acc=78.1%  |  val  loss=0.3708  acc=85.8%  f1=0.8804  prec=0.8953  rec=0.8660


  Epoch  3/20  |  train  loss=0.3047  acc=88.6%  |  val  loss=0.3173  acc=86.6%  f1=0.8870  prec=0.9033  rec=0.8713


  Epoch  4/20  |  train  loss=0.1993  acc=92.8%  |  val  loss=0.3245  acc=86.2%  f1=0.8834  prec=0.9030  rec=0.8647


  Epoch  5/20  |  train  loss=0.1231  acc=96.1%  |  val  loss=0.3555  acc=85.6%  f1=0.8797  prec=0.8896  rec=0.8700
  Early stopping triggered (no improvement for 2 epochs).



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 86.37%   F1 : 0.8858   Precision : 0.8980   Recall : 0.8738
──────────────────────────────────────────────────────────────



batch_loss,██▇▇▇▇█▇▇▆▆▆▆▅▅▅▄▄▄▃▂▂▃▂▂▂▁▂▂▂▂▁▁▁▁▂▁▂▂▁
epoch,▁▃▅▆█
train_accuracy,▁▄▇▇█
train_loss,█▆▃▂▁
val_accuracy,▁▇██▇
val_f1,▁▇██▇
val_loss,█▃▁▁▂
val_precision,▁███▇
val_recall,█▁▁▁▁
batch_loss,0.14431
epoch,5


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_ANN_Large.pth
[Result] Experiment       : Task01_ANN_Large
[Result] Best Val Accuracy: 86.58%
[Result] Test Accuracy    : 86.37%
[Result] Test F1          : 0.8858



In [3]:
# grade 5  Simple ANN — public dataset (~100 K reviews from amazon_polarity)
from experiments.task01_ann import main as run_ann

ann_public = run_ann(dataset="public")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading ANN data  [public dataset] …
  (This may take a while on first run — the dataset is ~1 GB)


  Loaded 3,600,000 examples from 'amazon_polarity'
  Preprocessing text (classical) …
  Fitting TF-IDF on training split (will NOT see val / test) …
  TF-IDF vocab: 30,000 features
  Split: 2,521,440 train / 538,560 val / 540,000 test

Building ANN  (vocab_size=30,000) …
  [build_ann] dataset='public' → LargeANN (3 blocks, 512→256→64)
  Total parameters    : 15,509,954
  Trainable parameters: 15,509,954



══════════════════════════════════════════════════════════════
  Experiment  : Task01_ANN_Public
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=0.0001
  Epochs      : 10
  Patience    : 2
  Trainable   : 15,509,954 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/10  |  train  loss=0.2871  acc=87.5%  |  val  loss=0.2286  acc=90.8%  f1=0.9080  prec=0.9059  rec=0.9101


  Epoch  2/10  |  train  loss=0.2448  acc=90.3%  |  val  loss=0.2208  acc=91.2%  f1=0.9119  prec=0.9090  rec=0.9148


...

  Epoch  9/10  |  train  loss=0.2048  acc=92.2%  |  val  loss=0.1959  acc=92.3%  f1=0.9227  prec=0.9268  rec=0.9187


  Epoch 10/10  |  train  loss=0.1885  acc=93.0%  |  val  loss=0.1948  acc=92.4%  f1=0.9230  prec=0.9319  rec=0.9144



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 92.46%   F1 : 0.9239   Precision : 0.9329   Recall : 0.9151
──────────────────────────────────────────────────────────────



batch_loss,█▃▃▂▂▂▂▃▂▂▃▂▂▂▂▂▂▃▃▃▂▂▃▃▂▂▂▁▃▁▂▂▁▃▂▃▃▃▂▂
epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▅▅▅▅▆▆▆▇█
train_loss,█▅▅▄▄▄▃▃▂▁
val_accuracy,▁▃▃▄▅▅▆▇██
val_f1,▁▃▃▄▅▆▆▇██
val_loss,█▆▆▅▅▄▃▂▁▁
val_precision,▁▂▄▃▃▄▆▆▇█
val_recall,▁▄▁▄█▅▅▆▆▄
batch_loss,0.11497
epoch,10


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_ANN_Public.pth
[Result] Experiment       : Task01_ANN_Public
[Result] Best Val Accuracy: 92.38%
[Result] Test Accuracy    : 92.46%
[Result] Test F1          : 0.9239



---
## Section 2 — Task 1.1: Bidirectional LSTM

A BiLSTM with **learned word embeddings** that reads the sentence in both directions.  
Unlike the ANN, the BiLSTM preserves word order and captures long-range dependencies  
(e.g. negations, emphasis) — important signals for sentiment.

| Cell | Dataset | What we learn |
|---|---|---|
| 2.1 | Small (1 K) | Sequence model on limited data |
| 2.2 | Large (25 K) | Sequence model vs ANN on the same 25 K data |

In [4]:
# 2.1  BiLSTM — small dataset (1 K reviews)
from experiments.task01_bilstm import main as run_bilstm

bilstm_small = run_bilstm(dataset="small")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading BiLSTM data  [small dataset] …
  Preprocessing text (classical) …
  Building vocabulary from training split only …
  Vocabulary size: 1,418 tokens (max 30,000)
  Split: 686 train / 147 val / 147 test

Building BiLSTMSentiment  (vocab_size=1,418) …
  Total parameters    : 2,550,018
  Trainable parameters: 2,550,018



══════════════════════════════════════════════════════════════
  Experiment  : Task01_BiLSTM_Small
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=0.001
  Epochs      : 20
  Patience    : 5
  Trainable   : 2,550,018 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/20  |  train  loss=0.6916  acc=49.4%  |  val  loss=0.6735  acc=62.6%  f1=0.7150  prec=0.5798  rec=0.9324


  Epoch  2/20  |  train  loss=0.6518  acc=62.0%  |  val  loss=0.6159  acc=64.6%  f1=0.5185  prec=0.8235  rec=0.3784


  Epoch  3/20  |  train  loss=0.5553  acc=71.0%  |  val  loss=0.5612  acc=70.7%  f1=0.6667  prec=0.7818  rec=0.5811


...

  Epoch 17/20  |  train  loss=0.1910  acc=91.8%  |  val  loss=0.5697  acc=75.5%  f1=0.7831  prec=0.7065  rec=0.8784


  Epoch 18/20  |  train  loss=0.1684  acc=92.9%  |  val  loss=0.7109  acc=76.2%  f1=0.7712  prec=0.7468  rec=0.7973
  Early stopping triggered (no improvement for 5 epochs).



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 74.15%   F1 : 0.7324   Precision : 0.7536   Recall : 0.7123
──────────────────────────────────────────────────────────────



batch_loss,████▆▇▅▅▅▄▄▅▄▅▄▂▃▃▂▃▃▂▂▂▂▂▁▁▂▃▂▁▂▃▂▁▂▁▂▂
epoch,▁▁▂▂▃▃▃▄▄▅▅▆▆▆▇▇██
train_accuracy,▁▃▄▅▅▆▆▆▇▇████████
train_loss,█▇▆▆▅▅▄▄▃▃▂▂▁▂▁▁▁▁
val_accuracy,▁▂▄▄▄▄▆▄▅▇▇▇█▇█▆▇▇
val_f1,▆▁▅▆▆▆▇▇▇▇▇████▇█▇
val_loss,▆▄▂▁▂▄▅▅█▅█▇▅▂▄▇▃▇
val_precision,▁█▇▄▄▄▅▃▄▆▆▇▇▇▇▄▅▆
val_recall,█▁▄▆▆▆▇▇▇▆▆▆▆▆▆▇▇▆
batch_loss,0.18553
epoch,18


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_BiLSTM_Small.pth
[Result] Experiment       : Task01_BiLSTM_Small
[Result] Best Val Accuracy: 78.91%
[Result] Test Accuracy    : 74.15%
[Result] Test F1          : 0.7324



In [ ]:
# 2.2  BiLSTM — large dataset (25 K reviews)
from experiments.task01_bilstm import main as run_bilstm

bilstm_large = run_bilstm(dataset="large")

In [ ]:
# grade 5  BiLSTM — public dataset (~100 K reviews from amazon_polarity)
from experiments.task01_bilstm import main as run_bilstm

bilstm_public = run_bilstm(dataset="public")

---
## Section 3 — Task 1.2: BERT Fine-Tuning

**BERT-base-uncased** was pre-trained on BookCorpus + English Wikipedia using  
masked language modelling and next-sentence prediction.  
We replace its classification head and fine-tune **all 12 transformer layers** for sentiment.

> ⚠ GPU strongly recommended — BERT has 110 M parameters.  
> On CPU, reduce `"epochs"` in `config.py → BERT_CONFIG` to `1`.

In [ ]:
# 3.1  BERT — fine-tuned on the large Amazon dataset (25 K reviews)
from experiments.task02_bert import main as run_bert

bert_results = run_bert()

---
## Section 4 — Task 1.2: DistilBERT Fine-Tuning

**DistilBERT-base-uncased** is a knowledge-distilled version of BERT:  
40% fewer parameters, 60% faster, ~97% of BERT's accuracy.  

Running both side-by-side provides a direct **complexity vs accuracy trade-off** study.

In [ ]:
# 4.1  DistilBERT — fine-tuned on the large Amazon dataset (25 K reviews)
from experiments.task02_distilbert import main as run_distilbert

distilbert_results = run_distilbert()

---
## Section 5 — Grade 5: Two Transformers on the Public Dataset (~1 GB)

Runs **BERT-base-uncased** and **DistilBERT-base-uncased** back-to-back,  
both fine-tuned on `amazon_polarity` from Hugging Face.

- **amazon_polarity** — ~3.6 M product reviews, ~1 GB download  
- Training is capped at `PUBLIC_MAX_SAMPLES` rows (set in `config.py`, default 100 K)  
- Both models train on the identical data split for a fair comparison

> ⚠ GPU strongly recommended.  On CPU, set `"epochs": 1` in
> `BERT_PUBLIC_CONFIG` and `DISTILBERT_PUBLIC_CONFIG` in `config.py`.

In [ ]:

# 5.1  BERT + DistilBERT on the public ~1 GB amazon_polarity dataset
from experiments.grade5_transformers_public import main as run_grade5

grade5_results = run_grade5()


---
## Section 6 — Task 1.3: Standardized Comparison (Required by Spec)

This section satisfies the Task 1.3 requirement:
> *"Use the **same test dataset** for Simple ANN, LSTM-based model and the Transformer."*

The cell below trains all three models on the **25 K large dataset**, evaluates them on the **exact same 15 % test split**, and prints:

- Test accuracy and F1 score for each model
- Parameter count (model complexity)
- Relative inference speed (compared to the ANN baseline)
- A written discussion of all three Task 1.3 comparison questions

The standardization is guaranteed by `RANDOM_SEED = 42` in `config.py` — every model's data loader applies the same stratified split to the same raw texts.

> ⚠ This cell trains all three models sequentially. Run Sections 1–5 first if you want individual results, or run this cell alone for the complete standardized comparison.

In [ ]:
# Task 1.3 — Standardized comparison: Simple ANN vs BiLSTM vs BERT on the identical test split
from experiments.task03_comparison import main as run_task03_comparison
comparison_results = run_task03_comparison()

---
## Section 6 — Model Comparison

Print a side-by-side table of all experiment results.  
Requires the cells above to have been run in the same session.

In [ ]:
# 6.1  Print side-by-side comparison of all models
results_dict = {}

try: results_dict["ANN  (small 1K)"]    = ann_small
except NameError: pass
try: results_dict["ANN  (large 25K)"]   = ann_large
except NameError: pass
try: results_dict["ANN  (public 100K)"] = ann_public
except NameError: pass
try: results_dict["BiLSTM (small 1K)"]  = bilstm_small
except NameError: pass
try: results_dict["BiLSTM (large 25K)"] = bilstm_large
except NameError: pass
try: results_dict["BiLSTM (public)"]    = bilstm_public
except NameError: pass
try: results_dict["BERT   (large 25K)"] = bert_results
except NameError: pass
try: results_dict["DistilBERT (25K)"]   = distilbert_results
except NameError: pass
try: results_dict["BERT   (public)"]    = grade5_results.get("BERT", {})
except (NameError, AttributeError): pass
try: results_dict["DistilBERT (pub)"]   = grade5_results.get("DistilBERT", {})
except (NameError, AttributeError): pass

print(f"{'Model':<25} {'Test Acc':>10}  {'Test F1':>10}  {'Best Val Acc':>12}")
print("-" * 62)
for name, res in results_dict.items():
    print(
        f"  {name:<23}"
        f"  {res.get('test_accuracy', float('nan')):>8.2f}%"
        f"  {res.get('test_f1', float('nan')):>10.4f}"
        f"  {res.get('best_val_accuracy', float('nan')):>10.2f}%"
    )
print("-" * 62)
print("\nView training curves at https://wandb.ai  (project: advanced-ai-lab-1)")

---
## Section 7 — Weights & Biases: View All Results

All experiments automatically stream metrics to **https://wandb.ai** — no local server needed.

### Viewing results
Open **https://wandb.ai** in any browser — all runs are grouped under project **`advanced-ai-lab-1`**.  
Use the **Charts** panel to compare training curves, val F1, and test accuracy side-by-side.

### Key comparisons to make
- **ANN small vs large** — impact of dataset size on bag-of-words features  
- **ANN vs BiLSTM** — bag-of-words vs sequence model on the same data  
- **BiLSTM vs BERT** — trained-from-scratch vs pre-trained transformer  
- **BERT vs DistilBERT** — accuracy trade-off for model compression  

> To rename the project, change `WANDB_PROJECT` in `config.py`.